# Can we predict whether a warranty claim is filed within a year?
Target: `claim_within_year` (1 = claim filed within a year of sale). Plan: inspect data quality, explore features, then compare a baseline against logistic regression and gradient boosting with cross-validation and a held-out test set. Claims are rare (~13%), so accuracy is misleading; we use ROC AUC and PR AUC (average precision) against the no-skill baseline.

In [ ]:
import os; os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
SEED = 0
df = pd.read_csv('data.csv')
pd.read_csv('columns.csv')

## 1. Data quality

In [ ]:
print(df.shape, '| missing:', int(df.isna().sum().sum()), '| duplicate rows:', df.duplicated().sum(), '| duplicate serials:', df.device_serial.duplicated().sum())
df.describe().T

In [ ]:
print('daily_hours > 24:', (df.daily_hours > 24).sum(), '| == 0:', (df.daily_hours == 0).sum())
print('device_age_years < 0.05:', (df.device_age_years < 0.05).sum(), '| price <= 0:', (df.price_eur <= 0).sum())
df[df.daily_hours > 24].head()

`daily_hours` has values above 24, impossible for a daily average (it is self-reported at registration). We keep those rows but cap at 24 and flag it as an issue for the data owner. `device_serial` is a unique identifier, so it is excluded from modelling.

## 2. Exploration

In [ ]:
print('Base rate:', df.claim_within_year.mean().round(4))
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
df.groupby('channel').claim_within_year.mean().plot.bar(ax=ax[0], title='Claim rate by channel', rot=0)
for a, c in zip(ax[1:], ['device_age_years', 'price_eur', 'daily_hours']):
    q = pd.qcut(df[c], 10, duplicates='drop')
    df.groupby(q, observed=True).claim_within_year.mean().plot(ax=a, marker='o', title=f'Claim rate by {c} decile')
    a.set_xticks([])
plt.tight_layout(); plt.show()
df.groupby('channel').claim_within_year.agg(['mean', 'count'])

## 3. Modelling setup
Stratified 75/25 split; the test set is used once at the end. Model comparison uses 5-fold stratified CV on the training portion.

In [ ]:
FEATS = ['channel', 'device_age_years', 'price_eur', 'daily_hours']
X = df[FEATS].copy(); X['daily_hours'] = X.daily_hours.clip(upper=24); y = df.claim_within_year
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, stratify=y, random_state=SEED)
num = ['device_age_years', 'price_eur', 'daily_hours']
lin_pre = ColumnTransformer([('c', OneHotEncoder(drop='first'), ['channel']), ('n', StandardScaler(), num)])
tree_pre = ColumnTransformer([('c', OneHotEncoder(), ['channel']), ('n', 'passthrough', num)])
models = {
    'baseline (prior)': DummyClassifier(strategy='prior'),
    'logistic': make_pipeline(lin_pre, LogisticRegression(max_iter=1000)),
    'grad. boosting': make_pipeline(tree_pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=SEED)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for n, m in models.items():
    r = cross_validate(m, Xtr, ytr, cv=cv, scoring=['roc_auc', 'average_precision'])
    rows.append({'model': n, 'AUC mean': r['test_roc_auc'].mean(), 'AUC sd': r['test_roc_auc'].std(),
                 'PR-AUC mean': r['test_average_precision'].mean(), 'PR-AUC sd': r['test_average_precision'].std()})
pd.DataFrame(rows).round(3)

## 4. Held-out test performance

In [ ]:
res = {}
for n, m in models.items():
    m.fit(Xtr, ytr); res[n] = m.predict_proba(Xte)[:, 1]
print('Test base rate (no-skill PR-AUC):', yte.mean().round(3))
print(pd.DataFrame({n: {'AUC': roc_auc_score(yte, p), 'PR-AUC': average_precision_score(yte, p)} for n, p in res.items()}).T.round(3))
rng = np.random.default_rng(SEED); p = res['logistic']; yv = yte.values
b = [roc_auc_score(yv[i], p[i]) for i in (rng.integers(0, len(yv), len(yv)) for _ in range(500))]
print('Logistic test AUC 95% bootstrap CI:', np.percentile(b, [2.5, 97.5]).round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n, p in res.items():
    f, t, _ = roc_curve(yte, p); ax[0].plot(f, t, label=n)
    pr, rc, _ = precision_recall_curve(yte, p); ax[1].plot(rc, pr, label=n)
ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR'); ax[1].set(title='Precision-recall', xlabel='Recall', ylabel='Precision'); ax[0].legend()
plt.tight_layout(); plt.show()

## 5. What drives the prediction?

In [ ]:
lr = models['logistic']
coef = pd.Series(lr[-1].coef_[0], index=lr[0].get_feature_names_out()).sort_values()
print('Logistic coefficients (log-odds; numerics standardised):'); print(coef.round(3))
pi = permutation_importance(models['grad. boosting'], Xte, yte, scoring='roc_auc', n_repeats=20, random_state=SEED)
pd.Series(pi.importances_mean, index=FEATS).sort_values().round(4).rename('AUC drop when permuted (GB)')

## 6. Is there a usable targeting rule?
Lift when flagging the highest-risk fraction of the test set (logistic scores).

In [ ]:
p = pd.Series(res['logistic'], index=yte.index)
rows = []
for frac in [.05, .1, .2, .3, .5]:
    top = p.nlargest(int(len(p) * frac)).index
    rows.append({'top share': frac, 'claim rate in group': yte[top].mean(), 'lift vs base': yte[top].mean() / yte.mean(),
                 'share of all claims captured': yte[top].sum() / yte.sum()})
pd.DataFrame(rows).round(3)

## Conclusions
- **Yes, moderately.** Held-out ROC AUC is ~0.74 (95% bootstrap CI ~0.71-0.77) and PR-AUC ~0.32 vs a 0.126 no-skill baseline. CV agrees (AUC 0.72 +/- 0.03). Useful for ranking, not for confident individual calls.
- **Drivers:** older devices, higher reported daily hours and higher price all raise claim risk; `device_age_years` is strongest. Channel matters little (partner is somewhat higher; online vs store is indistinguishable). These are associations, not causes.
- **Model choice:** plain logistic regression matches or beats gradient boosting, so the signal looks mostly monotone/additive; no need for a complex model.
- **Practical use:** flagging the riskiest 10% of warranties gives a ~37% claim rate (about 3x lift) and captures ~30% of all claims; the riskiest 30% captures ~60%.
- **Caveats:** 23 rows have `daily_hours` > 24 (impossible; capped), and `daily_hours` is self-reported. Only 4 features and 7,000 rows, so the ceiling is limited. Single random split, with no time-based validation, so performance on future sales is untested. Calibration was not evaluated, so use scores for ranking, not as probabilities.
- **Execution note:** outputs were generated by running the cells in a plain Python process, since Jupyter kernels could not start in the sandbox. Re-running in Jupyter should give the same results (fixed seeds).